In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import re
from IPython.display import display
from datetime import datetime
from operator import attrgetter
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import unicodedata
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from sklearn.cluster import AgglomerativeClustering
from sklearn.mixture import GaussianMixture

## Credit Membres

In [4]:
CreditMembres = pd.read_excel('00-Original/BBDDCreditMembres_origin.xlsx', header=None)
CreditMembres.columns = CreditMembres.iloc[2]
CreditMembres = CreditMembres.drop([0, 1, 2]).reset_index(drop=True)
CreditMembres = CreditMembres.drop(columns=["Peça", "Tipologia roba quantitats"])
CreditMembres = CreditMembres.drop(columns=[col for col in CreditMembres.columns if pd.isna(col)])

CreditMembres = CreditMembres[CreditMembres["Membre"] != 'Not found']
CreditMembres = CreditMembres.rename(columns={
    "Membre" : "membre",
    "PEÇA": "peca",
    "U": "unitats"
})

CreditMembres['membre'] = CreditMembres['membre'].str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)
CreditMembres['peca'] = CreditMembres['peca'].str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)

def quitar_acentos(texto):
    if pd.isnull(texto):
        return texto
    return ''.join(
        c for c in unicodedata.normalize('NFKD', str(texto))
        if not unicodedata.combining(c)
    )

def limpiar_texto(texto):
    if pd.isnull(texto):
        return texto
    texto = str(texto).lower()
    texto = unicodedata.normalize('NFKD', texto)
    texto = ''.join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r'\s+', ' ', texto)  # Espacios múltiples → uno solo
    return texto.strip()

CreditMembres['membre'] = CreditMembres['membre'].apply(quitar_acentos)
CreditMembres['membre'] = CreditMembres['membre'].apply(limpiar_texto)

CreditMembres['peca'] = CreditMembres['peca'].apply(quitar_acentos)
CreditMembres['peca'] = CreditMembres['peca'].apply(limpiar_texto)

CreditMembres["unitats"] = pd.to_numeric(CreditMembres["unitats"], errors="coerce").astype("Int64")

CreditMembres = CreditMembres[CreditMembres["unitats"] != 0]

CreditMembres['peca'] = CreditMembres['peca'].replace({
    'camiseta dr maniga curta': 'samarreta maniga curta',
    'samarreta de maniga curta': 'samarreta maniga curta',
    'samarreta de màniga llarga': 'samarreta maniga llarga',
    'vestido': 'vestit',
    'Vestit' : 'vestit',
    'short': 'pantalons curts',
    'botas': 'botes',
    'sandalias': 'sandalies',
    'Americana':'americana',
    'chaleco': 'armilla',
    'top,samarreta': 'samarreta'
})

CreditMembres = CreditMembres[~CreditMembres['peca'].isna()]

tops = ['abric', 'americana', 'armilla', 'body', 'camisa', 'camisa de màniga curta', 'samarreta', 'jaqueta', 'jersei', 'polo', 'samarreta maniga curta', 'samarreta maniga llarga', 'samarreta sense mànigues', 'sudadera', 'top']
bottoms = ['faldilla','pantalons curts', 'pantalons llargs',]
enteros = ['vestit', 'peto', 'mono']
sabates = ['botas de caña', 'botes', 'botines', 'sabates', 'sabates de taló', 'sandalies', 'bambas']
accesorios = ['bolso', 'cinturo', 'motxilla']
desconegut = ['desconegut']

def clasificar_categoria(peca):
    if any(word in peca for word in tops):
        return 'tops'
    elif any(word in peca for word in bottoms):
        return 'bottoms'
    elif any(word in peca for word in enteros):
        return 'enteros'
    elif any(word in peca for word in sabates):
        return 'sabates'
    elif any(word in peca for word in accesorios):
        return 'accesorios'
    else:
        return 'otros'  # Por si hay algo no clasificado

CreditMembres['categoria'] = CreditMembres['peca'].apply(clasificar_categoria)

CreditMembres = CreditMembres[['membre', 'categoria', 'peca', 'unitats']]

CreditMembres = CreditMembres.sort_values('membre', ascending=True).reset_index(drop=True)

In [5]:
CreditMembres

2,membre,categoria,peca,unitats
0,a fernandez,tops,jersei,1
1,a fernandez,bottoms,pantalons llargs,1
2,adria linares i prenafeta,tops,polo,1
3,adria linares i prenafeta,tops,samarreta,7
4,adria linares i prenafeta,tops,jersei,2
...,...,...,...,...
1511,youssef charkaoui koubaa,bottoms,pantalons llargs,2
1512,youssef charkaoui koubaa,tops,camisa de maniga curta,1
1513,youssef charkaoui koubaa,tops,camisa,1
1514,youssef charkaoui koubaa,tops,jaqueta,1


In [84]:
CreditMembres.to_csv("CreditMembres_treated.csv")

In [56]:
result = CreditMembres.groupby('membre')['unitats'].sum()
result_df = result.reset_index().sort_values(by='unitats', ascending=True)
result_df.to_csv("resumen_unitats_por_membre.csv")

## Evolucio Membres

In [7]:
EvolucioMembres = pd.read_excel('00-Original/BBDDEvolucioMembres_origin.xlsb')

EvolucioMembres = EvolucioMembres.drop(columns=["INFO IMPORTANT", "Unnamed: 10", "Start Date (UTC)", "Current Period Start (UTC)", "Quantity", "Currency", "Interval"])

EvolucioMembres = EvolucioMembres.rename(columns={
    "TIPUS DE QUOTA": "quota",
    "Status": "status",
    "ALTA": "alta",
    "ÚLTIM PAGAMENT": "ultim_pagament",
    "Customer Name": "membre"
})

EvolucioMembres = EvolucioMembres[['membre', 'status', 'alta', 'ultim_pagament', 'quota']]

EvolucioMembres['membre'] = EvolucioMembres['membre'].str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)

def quitar_acentos(texto):
    if pd.isnull(texto):
        return texto
    return ''.join(
        c for c in unicodedata.normalize('NFKD', str(texto))
        if not unicodedata.combining(c)
    )

def limpiar_texto(texto):
    if pd.isnull(texto):
        return texto
    texto = str(texto).lower()
    texto = unicodedata.normalize('NFKD', texto)
    texto = ''.join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r'\s+', ' ', texto)  # Espacios múltiples → uno solo
    return texto.strip()

EvolucioMembres['membre'] = EvolucioMembres['membre'].apply(quitar_acentos)
EvolucioMembres['membre'] = EvolucioMembres['membre'].apply(limpiar_texto)

EvolucioMembres = EvolucioMembres[EvolucioMembres['quota'] != 0.5]

EvolucioMembres['alta'] = pd.to_datetime(EvolucioMembres['alta'], unit='d', origin='1899-12-30')
EvolucioMembres['ultim_pagament'] = pd.to_datetime(EvolucioMembres['ultim_pagament'], unit='d', origin='1899-12-30')
EvolucioMembres['membre'] = EvolucioMembres['membre'].replace({
    r'√(†|¢|°|≈†)': 'a',  
    r'√©': 'e',
    r'√(≠|=)': 'i',
    r'√(≥|ö|∂|o)': 'o',
    r'√±': 'n',
    r'√∩': 'u',
    r'[≈†]': ''
}, regex=True)

EvolucioMembres_IncompleteExpired = EvolucioMembres[EvolucioMembres['status'] == 'incomplete_expired']
EvolucioMembres_IncompleteExpired = EvolucioMembres_IncompleteExpired.drop_duplicates(subset='membre', keep='last')

conteo_status = EvolucioMembres.groupby('membre')['status'].nunique()
nombres_unico_status = conteo_status[conteo_status == 1].index
ExclusiveIncomplete = EvolucioMembres[(EvolucioMembres['membre'].isin(nombres_unico_status)) & (EvolucioMembres['status'] == 'incomplete_expired')]
ExclusiveIncomplete = ExclusiveIncomplete['membre'].unique()
ExclusiveIncomplete_df = EvolucioMembres[EvolucioMembres['membre'].isin(ExclusiveIncomplete)]
ExclusiveIncomplete_df = ExclusiveIncomplete_df.drop_duplicates(subset='membre', keep='last')
ExclusiveIncomplete_df

EvolucioMembres = EvolucioMembres[EvolucioMembres['status'] != 'incomplete_expired']
EvolucioMembres.sort_values(by='membre', ascending=True)
EvolucioMembres = pd.concat([EvolucioMembres, ExclusiveIncomplete_df], ignore_index=True)

EvolucioMembres = EvolucioMembres.sort_values(by=['membre', 'alta'], ascending=True).reset_index(drop=True)

In [158]:
MembrosMultiplasAltas = EvolucioMembres[EvolucioMembres['membre'].duplicated(keep=False)]
MembrosMultiplasAltas.to_csv("EvolucioMembres_MembresMultiplesAltes.treated.csv")

In [8]:
EvolucioMembres

,membre,status,alta,ultim_pagament,quota
0,a fernandez,canceled,2024-12-09 19:05:00.000000139,2025-06-09 19:05:00.000000139,10.0
1,adria linares i prenafeta,active,2024-12-21 18:01:59.999999759,2025-07-21 18:01:59.999999759,10.0
2,adriana alarcon torrella,active,2025-01-09 18:24:59.999999860,2025-07-09 18:24:59.999999860,10.0
3,adriana iri,active,2024-11-09 16:05:59.999999703,2025-07-09 16:05:59.999999703,10.0
4,agnes fauro sierra,active,2024-11-08 17:17:59.999999953,2025-07-08 17:17:59.999999953,10.0
...,...,...,...,...,...
254,wendy fajardo,canceled,2025-02-15 16:43:00.000000027,2025-07-15 16:43:00.000000027,10.0
255,xavi perez,active,2024-11-15 19:34:00.000000157,2025-07-15 19:34:00.000000157,10.0
256,xenia antras agut,active,2024-11-13 17:51:59.999999693,2025-07-13 17:51:59.999999693,10.0
257,xiaohan shi,canceled,2025-04-23 16:18:00.000000167,2025-05-23 16:18:00.000000167,10.0


In [159]:
EvolucioMembres.to_csv("EvolucioMembres.treated.csv")

## Membres Actius

In [9]:
MembresActius = pd.read_excel('00-Original/BBDDMembresActius_origin.xlsx', header=None)
MembresActius.columns = MembresActius.iloc[2]
MembresActius = MembresActius.drop([0, 1, 2]).reset_index(drop=True)
MembresActius = MembresActius.dropna(how='all')
MembresActius = MembresActius.drop(columns=["Phone", "Items", "Nivell", "."])
MembresActius = MembresActius.drop(columns=[col for col in MembresActius.columns if pd.isna(col)])
MembresActius = MembresActius.rename(columns={
    "Membre":"membre",
    "Alta":"alta",
    "Email":"email",
    "Codi Postal":"codipostal",
    "Població":"poblacio",
    "Estat":"status",
    "Antiguitat":"antiguitat",
    "Baixa":"baixa"
})

MembresActius['membre'] = MembresActius['membre'].str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)

def quitar_acentos(texto):
    if pd.isnull(texto):
        return texto
    return ''.join(
        c for c in unicodedata.normalize('NFKD', str(texto))
        if not unicodedata.combining(c)
    )

def limpiar_texto(texto):
    if pd.isnull(texto):
        return texto
    texto = str(texto).lower()
    texto = unicodedata.normalize('NFKD', texto)
    texto = ''.join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r'\s+', ' ', texto)  # Espacios múltiples → uno solo
    return texto.strip()

MembresActius['membre'] = MembresActius['membre'].apply(quitar_acentos)
MembresActius['membre'] = MembresActius['membre'].apply(limpiar_texto)
MembresActius['poblacio'] = MembresActius['poblacio'].apply(quitar_acentos)
MembresActius['poblacio'] = MembresActius['poblacio'].apply(limpiar_texto)

MembreDonatBaixa = MembresActius[MembresActius['status']!='active']
MembreDonatBaixa= MembreDonatBaixa.reset_index(drop=True).sort_values(by='membre',ascending=True)

MembresActius = MembresActius[MembresActius['status']!='canceled']
MembresActius= MembresActius.reset_index(drop=True).sort_values(by='antiguitat',ascending=False)

MembresActius = pd.concat([MembreDonatBaixa, MembresActius], ignore_index=True)

In [11]:
MembresActius

2,membre,email,codipostal,poblacio,status,antiguitat,alta,baixa
0,aryane sanches dourado leao,natureenergymassage@gmail.com,8041,barcelona,canceled,Client donat de baixa,"13/11/2024, 7:56 PM",6/24/2025
1,bernardo kleinfinger,ber.klein2@gmail.com,8015,barcelona,canceled,Client donat de baixa,"26/03/2025, 7:49 AM",6/24/2025
2,carlos sariola aznar,carlis.sariola@gmail.com,8012,barcelona,canceled,Client donat de baixa,"14/02/2025, 1:24 PM",2025-07-07 00:00:00
3,fanny u,faney@hotmail.fr,8024,barcelona,canceled,Client donat de baixa,"17/06/2025, 1:57 PM",2025-06-07 00:00:00
4,gemma graells,gemmagraells@gmail.com,8024,barcelona,canceled,Client donat de baixa,"08/05/2025, 5:15 PM",2025-01-07 00:00:00
...,...,...,...,...,...,...,...,...
145,maria fernanda corona manon,fernandacoronam3@gmail.com,8008,barcelona,active,1 month,"05/06/2025, 7:08 PM",NaN
146,isabel mendez,abgimendez@gmail.com,8012,barcelona,active,1 month,"02/06/2025, 7:47 PM",NaN
147,kehren jeanne barbour,kehrenbarbour@gmail.com,8012,barcelona,active,1 month,"30/05/2025, 1:20 PM",NaN
148,luigi nicola diana,luiginicola23@gmail.com,8012,barcelona,active,1 month,"29/05/2025, 6:48 PM",NaN


In [10]:
MembresActius.to_csv("MembresActius_treated.csv")

## Visites i canvis

In [12]:
VisitesiCanvis = pd.read_excel('00-Original/BBDDVisitesiCanvis_origin.xlsx', header=None)
VisitesiCanvis.columns = VisitesiCanvis.iloc[2]
VisitesiCanvis = VisitesiCanvis.drop([0, 1, 2]).reset_index(drop=True)
VisitesiCanvis = VisitesiCanvis.drop(columns=[col for col in VisitesiCanvis.columns if pd.isna(col)])
VisitesiCanvis = VisitesiCanvis.dropna(how='all')
VisitesiCanvis = VisitesiCanvis[VisitesiCanvis['Membre'] != 'Not found']
VisitesiCanvis = VisitesiCanvis.drop(columns=["Atès per", "Month name of Data de la Visita", "Botiga", "Comentaris", "TOP 5"])
VisitesiCanvis = VisitesiCanvis.rename(columns={
    "Membre":"membre",
    "Data de la Visita": "data_visita",
    "Nº de Canvis": "qtd_canvis"
})

VisitesiCanvis['data_visita'] = pd.to_datetime(VisitesiCanvis['data_visita'], dayfirst=True)

VisitesiCanvis['membre'] = VisitesiCanvis['membre'].str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)

def quitar_acentos(texto):
    if pd.isnull(texto):
        return texto
    return ''.join(
        c for c in unicodedata.normalize('NFKD', str(texto))
        if not unicodedata.combining(c)
    )

def limpiar_texto(texto):
    if pd.isnull(texto):
        return texto
    texto = str(texto).lower()
    texto = unicodedata.normalize('NFKD', texto)
    texto = ''.join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r'\s+', ' ', texto)  # Espacios múltiples → uno solo
    return texto.strip()

VisitesiCanvis['membre'] = VisitesiCanvis['membre'].apply(quitar_acentos)
VisitesiCanvis['membre'] = VisitesiCanvis['membre'].apply(limpiar_texto)

VisitesiCanvis = VisitesiCanvis.drop_duplicates()
VisitesiCanvis= VisitesiCanvis.sort_values(by='data_visita', ascending=False).reset_index(drop=True)
VisitesiCanvis = VisitesiCanvis[VisitesiCanvis['membre'].notna()]

In [13]:
VisitesiCanvis

2,membre,data_visita,qtd_canvis
0,kenia regis silva,2025-07-04 19:24:00,6
1,evelyn arevalo ruano,2025-07-04 18:59:00,1
2,emily skipper,2025-07-04 18:52:00,4
3,roberta causa,2025-07-04 18:48:00,2
4,judith ibanez,2025-07-03 19:25:00,8
...,...,...,...
409,eirini kolliopoulou,2024-09-19 19:38:00,NaN
410,vera alejandra stojanov,2024-09-19 18:58:00,1
411,anna casas tortosa,2024-09-16 19:19:00,1
412,laia haiyang sala rodo,2024-09-16 18:56:00,1


In [233]:
VisitesiCanvis.to_csv("VisitesiCanvis_treated.csv")